# Working with ecoinvent

Notebooks 01 and 02 used a toy background of three processes. This one uses a real one: ecoinvent 3.12, cut-off system model, with 26,533 datasets, and the impact methods of ecoinvent's own LCIA workbook.

**You need an ecoinvent licence.** The notebook reads your own copy of ecoinvent's files. ecoinvent data is licensed, so this notebook is committed **without outputs**: run it on your machine to see them, and clear the outputs before you commit (`python notebooks/refresh_outputs.py --clear-licensed`).

You will:

1. import the ecoinvent files once, into a frozen release,
2. look inside the release with SQL,
3. import EF 3.1 and the IPCC 2021 carbon indicators from ecoinvent's LCIA workbook, as three methods,
4. load the background with the three methods together,
5. find a dataset, read its footprint, and ask where it is emitted, what it buys and which lines move it,
6. write a small database on ecoinvent and calculate it,
7. score a table of products in one go.

[00-getting-started](00-getting-started.ipynb) lists every function used here. Words like *release*, *market* and *cut-off* are in the [glossary](../docs/appendix/glossary.md).

## Set up

Three environment variables matter. Set them before you start Jupyter or VS Code:

| Variable | What it is | Example |
|---|---|---|
| `ECOINVENT_DIR` | your ecoinvent 3.12 cut-off ecoSpold2 folder, the one with `datasets/` and `MasterData/` | `~/ecoinvent/3.12-cutoff` |
| `ECOINVENT_LCIA` | ecoinvent's LCIA implementation workbook for 3.12 | `~/ecoinvent/3.12-lcia/LCIA Implementation 3.12.xlsx` |
| `HEADWATER_HOME` | optional: where releases are kept, in `$HEADWATER_HOME/releases`. Default `~/.headwater` | `~/lca/headwater` |

The cell stops with a message if one of the first two is missing. Method folders and the database are written to a temporary folder, never into this repository.

The helper `headwater(...)` runs the `headwater` command as in a terminal and prints what it prints.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import headwater as hw
from headwater.release import releases_dir

missing = [name for name in ["ECOINVENT_DIR", "ECOINVENT_LCIA"] if not os.environ.get(name)]
if missing:
    raise RuntimeError(
        f"Set {' and '.join(missing)} (see the table above), then restart the kernel."
    )
ecoinvent_dir = Path(os.environ["ECOINVENT_DIR"]).expanduser()
lcia_workbook = Path(os.environ["ECOINVENT_LCIA"]).expanduser()
assert (ecoinvent_dir / "datasets").is_dir(), f"no datasets/ folder in {ecoinvent_dir}"
assert lcia_workbook.is_file(), f"{lcia_workbook} is not a file"

work = Path(tempfile.mkdtemp(prefix="headwater-ecoinvent-"))
HEADWATER = shutil.which("headwater", path=str(Path(sys.executable).parent))


def headwater(*args):
    """Run the headwater command and print what it prints."""
    done = subprocess.run([HEADWATER, *map(str, args)], capture_output=True, text=True, cwd=work)
    print((done.stdout + done.stderr).rstrip())
    done.check_returncode()


print("releases are kept in", releases_dir())

## Import ecoinvent once

`headwater release import` reads every dataset, checks every line against `MasterData/`, links each input to the process that supplies it, checks units and roles, and freezes the result as Parquet files in a folder named by the hash of its content. You do this once per ecoinvent version. After that, nothing reads the ecoSpold2 files again, and every database on the machine uses the same release.

`--version` is required: ecoSpold2 files do not record the database version, and the version goes into every process id, `ecoinvent:3.12:cutoff:<activity uuid>_<product uuid>`, so ids of two versions never mix. ecoinvent 3.12 writes release 3.0 in every file, so the version cannot be checked against them: the import takes it as given and warns `version-unchecked`.

If the cache already holds an `ecoinvent-3.12-cutoff` release, the cell uses it.

In [ ]:
cached = sorted(
    releases_dir().glob("ecoinvent-3.12-cutoff@*"), key=lambda path: path.stat().st_mtime
)
if cached:
    print(f"Using {cached[-1].name}, the newest of {len(cached)} in the cache")
else:
    headwater("release", "import", ecoinvent_dir, "--version", "3.12")
    cached = sorted(releases_dir().glob("ecoinvent-3.12-cutoff@*"))
release_ref = cached[-1].name
release_ref

`release_ref` is the release's name and hash, such as `ecoinvent-3.12-cutoff@b9428cdd8045`. That is what a database writes in `background:` and what you give `hw.load`.

## Look inside the release

A release is a folder of Parquet files and a `release.json` that holds their hashes. It is never edited. `Release.open` checks every file, and `connect()` opens a DuckDB connection with one view per file, so you can ask questions in SQL. First, how big is it?

In [ ]:
from headwater.release import Release

release = Release.open(cached[-1])
con = release.connect()
sizes = {
    table: con.sql(f"select count(*) from {table}").fetchone()[0]
    for table in ["processes", "flows", "exchanges", "issues", "geographies"]
}
sizes

Each dataset has a `kind`, ecoinvent's activity type. A **market** mixes the suppliers of one product in one region, with transport and losses. `.df()` turns the answer into a pandas table.

In [ ]:
con.sql("""
    select kind, count(*) as datasets
    from processes
    group by kind
    order by datasets desc
""").df()

The `issues` table holds what the import resolved on purpose, such as data gaps. A release cannot be frozen while an issue is open, so every row here has a status and a reason.

In [ ]:
con.sql("select status, count(*) as lines from issues group by status").df()

## Import the impact methods

ecoinvent ships its methods as an Excel workbook, the LCIA implementation. `headwater method import` reads one method of it, finds each factor's elementary flow in the release by name and compartment, and writes a **method folder**: a `method.yaml` and a `factors.csv` with flow ids. It reports how many factors mapped, and lists every flow the method has no factor for.

In Brightway each category of a method is its own tuple, such as `("EF v3.1", "climate change", "global warming potential (GWP100)")`. In headwater one method holds several **indicators**, one per category.

We import three methods, each into the temporary folder with `--out`:

- **EF v3.1**, whole: 25 indicators.
- **IPCC 2021 (incl. biogenic CO2)**, three of its categories: biogenic removals, biogenic emissions, and their sum. EF 3.1 counts no biogenic CO₂.
- **IPCC 2021**, one category: direct land-use change.

Reporting standards ask for those carbon indicators apart. Why these three, and how they compare with ecoinvent's own scores, is in [docs/design.md](../docs/design.md#carbon-indicators).

In [ ]:
methods = work / "methods"
headwater(
    "method",
    "import",
    lcia_workbook,
    "--method",
    "EF v3.1",
    "--flow-list",
    release_ref,
    "--out",
    methods / "EF-3.1",
)

In [ ]:
headwater(
    "method",
    "import",
    lcia_workbook,
    "--method",
    "IPCC 2021 (incl. biogenic CO2)",
    "--category",
    "climate change: biogenic removals (incl. CO2)",
    "--category",
    "climate change: biogenic emissions (incl. CO2)",
    "--category",
    "climate change: biogenic (incl. CO2)",
    "--flow-list",
    release_ref,
    "--out",
    methods / "IPCC-2021-incl-biogenic-CO2",
)
headwater(
    "method",
    "import",
    lcia_workbook,
    "--method",
    "IPCC 2021",
    "--category",
    "climate change: direct land use change",
    "--flow-list",
    release_ref,
    "--out",
    methods / "IPCC-2021",
)

Each method is now a folder of two text files:

In [ ]:
print((methods / "IPCC-2021/method.yaml").read_text())

## Load the background with the three methods

`hw.load` takes the release and a list of method folders. It builds the matrices from the Parquet files, checks the lines, factorizes the matrix once, and solves the score of every product for every indicator of the three methods. This is the step that does the real work, once per notebook. [Notebook 05](05-brightway-and-headwater-on-ecoinvent.ipynb) times it.

In [ ]:
method_folders = [
    methods / "EF-3.1",
    methods / "IPCC-2021-incl-biogenic-CO2",
    methods / "IPCC-2021",
]
bg = hw.load(release_ref, method=method_folders)
print("method:", bg.method.id)
for indicator in bg.method.indicators:
    print(f"  {indicator.id:45} {indicator.unit}")

The three methods are used together under one name, their ids joined by `+`. Each keeps its own id, version and hash, and a result says which method its indicator came from.

A flow of ecoinvent that none of the methods covers counts 0, and is listed on `bg.not_characterised` with every process that uses it, so nothing is dropped without a trace:

In [ ]:
print(
    len(bg.not_characterised),
    "elementary flows are not covered by any of the methods, for example:",
)
for flow in bg.not_characterised[:3]:
    print(f"  {flow.name} ({flow.flow_id}), used by {len(flow.processes)} processes")

## Find a dataset

`bg.find` searches the names of the processes, then of their products, by exact rules: the name is the text, starts with it, or has every word of it. `location` keeps one place. The `process_id` is what a database line writes as a supplier. `headwater find` does the same from the command line, without loading anything.

In [ ]:
for match in bg.find("market for textile, woven cotton", limit=5):
    print(match.process_id)
    print(f"  {match.name} | {match.reference_product} | {match.location} | {match.unit}")
    print(f"  ({match.why})")
textile = bg.find("market for textile, woven cotton", location="GLO", limit=1)[0]

In [ ]:
headwater(
    "find",
    "electricity, medium voltage",
    "--release",
    release_ref,
    "--location",
    "FR",
    "--limit",
    "3",
)
electricity = bg.find("market for electricity, medium voltage", location="FR", limit=1)[0]
print(electricity.process_id)

## The footprint of one dataset

The background already holds the score of every product for every indicator. `hw.score_many` reads them for a table of products and amounts. With one row of 1 kg, it gives the footprint of 1 kg of woven cotton textile for all 29 indicators. `.to_arrow().to_pandas()` makes it a table; `.T` turns it so each indicator is a row.

In [ ]:
one_kg = hw.score_many(
    bg, {"sku": ["1 kg textile"], "product_id": [textile.process_id], "amount": [1.0]}
)
footprint = one_kg.to_arrow().to_pandas().set_index("sku").T
footprint["unit"] = one_kg.units
footprint

## Where is it emitted?

`bg.where` names the processes that emit the score, anywhere upstream, and sums them by location. With several indicators you name one: here EF 3.1's total climate change. Each process's part is its own direct emissions times how much it runs for 1 kg of textile.

In [ ]:
print(bg.where(textile.process_id, indicator="climate-change", top=10))

On 10 products of ecoinvent 3.12, for climate change and land use, these top 10 processes are the ones Brightway's contribution analysis lists, in the same order, in 20 of 20 checks ([benchmarks](../docs/appendix/benchmarks.md#top-contributors-against-brightway)).

## What does it buy?

`bg.contributions` shows the other view: what 1 kg of textile buys, and what each purchase is worth, as a tree. `min_share` folds purchases under that share into `rest`, and `max_depth` stops the tree.

In [ ]:
print(bg.contributions(textile.process_id, indicator="climate-change", min_share=0.05, max_depth=3))

## Which lines move it most?

`bg.sensitivity` ranks every line of every process upstream by elasticity: the % change of the score per 1% change of the line. This cell looks up each process's name with its location, and each flow's name, in the release's tables.

In [ ]:
process_names = dict(con.sql("select id, name || ' | ' || location from processes").fetchall())
flow_names = dict(con.sql("select id, name from flows").fetchall())
for line in bg.sensitivity(textile.process_id, indicator="climate-change", top=10):
    print(
        f"{line.elasticity:7.4f}  {process_names[line.process_id]}\n"
        f"         line {line.line_id}: {line.role.value} {line.amount:.4g} {line.unit} "
        f"of {flow_names[line.flow_id]}"
    )

## A small database on ecoinvent

Now a tote bag on ecoinvent. It buys 0.15 kg of woven cotton textile, 0.1 kWh of French medium-voltage electricity and 3 g of packaging film, and sends 0.01 kg of textile scraps to treatment. The suppliers are ids found with `find`. The database pins the release in `background:` and lists the three methods in `methods:`, in the order `hw.load` was given them. Its process `tote:tote` is scored per tote bag.

The scraps go to a market for waste textile. Its product is a waste, so the line's role is `waste_to_treatment` with a positive amount: kg sent. The film is written in `g`; headwater converts it to the flow's kg.

In [ ]:
film = bg.find("market for packaging film, low density polyethylene", location="GLO", limit=1)[0]
scraps = bg.find("market for waste textile, soiled", location="RoW", limit=1)[0]
flows = dict(con.sql("select p.id, p.reference_flow_id from processes p").fetchall())

database_folder = work / "databases/tote"
database_folder.mkdir(parents=True)
(database_folder / "database.yaml").write_text(f"""\
name: tote
background: {release_ref}
methods: [EF-3.1, IPCC-2021-incl-biogenic-CO2, IPCC-2021]
description: One tote bag on ecoinvent 3.12, a teaching example
""")
(database_folder / "flows.csv").write_text(
    "id,name,kind,unit\ntote:tote-bag,tote bag,product,unit\n"
)
(database_folder / "processes.csv").write_text(
    "id,name,location,kind\ntote,tote bag,FR,ordinary_transforming_activity\n"
)
(database_folder / "exchanges.csv").write_text(f"""\
process,line,role,flow,amount,unit,supplier
tote,e01,reference,tote:tote-bag,1,unit,
tote,e02,input,{flows[textile.process_id]},0.15,kg,{textile.process_id}
tote,e03,input,{flows[electricity.process_id]},0.1,kWh,{electricity.process_id}
tote,e04,input,{flows[film.process_id]},3,g,{film.process_id}
tote,e05,waste_to_treatment,{flows[scraps.process_id]},0.01,kg,{scraps.process_id}
""")
print((database_folder / "exchanges.csv").read_text())

Check it against the background, then calculate climate change. `db.validate(bg)` gives a report of errors and warnings; an empty report means `calc` will not refuse the database.

In [ ]:
db = hw.open(database_folder)
print(db.validate(bg))
result = db.calc(bg, "tote:tote", indicator="climate-change")
print(result)
print(result.contributions(min_share=0.05, max_depth=2))

`result.where()` names the processes that emit the tote's score. `result.sensitivity()` ranks the tote's own lines with every line upstream: the first lines of the tote show which of your own amounts matter most.

In [ ]:
print(result.where(top=5))
for line in result.sensitivity(top=5):
    name = process_names.get(line.process_id, line.process_id)
    print(f"{line.elasticity:7.4f}  {name} line {line.line_id}")

`db.scores(bg, "tote:tote")` gives every indicator at once. The biogenic sum can cancel: a product that takes CO₂ from the air and gives it back scores near 0. `provenance.cancellation` says so, for one indicator.

In [ ]:
scores = db.scores(bg, "tote:tote")
for indicator in bg.method.indicators:
    print(f"{indicator.id:45} {scores[indicator.id]:12.6g} {indicator.unit}")
biogenic = db.calc(bg, "tote:tote", indicator="climate-change-biogenic-incl-co2")
parts = biogenic.provenance.cancellation
print("from method:", biogenic.provenance.indicator_method.id)
print(f"positive part {parts.positive:.6g}, negative part {parts.negative:.6g}")
print("cancels:", parts.cancels)

The result lists the flows of the background that the tote reaches and no method counts:

In [ ]:
print(len(result.not_characterised), "flows reached with no factor, for example:")
for flow in result.not_characterised[:3]:
    print(f"  {flow.name}: {len(flow.processes)} processes that run for the tote")

## Many products at once

A **SKU** is one product you sell, described by what it buys from the background. `hw.score_many` scores a whole table in one multiplication with the stored scores. The table has one row per SKU and product: `sku`, `product_id` (the id of the process that makes the product) and `amount` (in the product's unit, kg sent for a waste). The amounts below are made up.

The stored scores can also be frozen as an **indicators release**, with `bg.write_indicators`. It needs no factorization to read, so a service that only scores SKUs never loads the matrix. Here it goes to the temporary folder; by default it goes beside the release in the cache.

In [ ]:
table = {
    "sku": ["tshirt-s", "tshirt-s", "tshirt-l", "tshirt-l", "tote", "tote"],
    "product_id": [
        textile.process_id,
        electricity.process_id,
        textile.process_id,
        electricity.process_id,
        textile.process_id,
        film.process_id,
    ],
    "amount": [0.18, 0.4, 0.26, 0.5, 0.15, 0.003],
}
indicators = bg.write_indicators(into=work / "releases")
print(indicators)
from_release = hw.score_many(
    indicators, table, indicators=["climate-change", "climate-change-biogenic-incl-co2"]
)
from_release.to_arrow().to_pandas()

A SKU score records the release and the methods it came from, by hash, so anyone who reads it knows which data produced it:

In [ ]:
print("releases:", [f"{r.name}@{r.hash[:12]}" for r in from_release.releases])
print("method:  ", from_release.method, from_release.method_version)

## What you did

- Imported ecoinvent 3.12 once into a release, and read it with SQL.
- Imported EF 3.1 and the IPCC 2021 carbon indicators from ecoinvent's workbook, and loaded them together.
- Found a dataset, read its footprint, where it is emitted, what it buys and which lines move it.
- Wrote a database on ecoinvent and calculated its tote bag for every indicator.
- Scored a table of SKUs from frozen indicator vectors.

Before you commit this notebook, clear its outputs: `uv run --group notebooks python notebooks/refresh_outputs.py --clear-licensed`. They hold ecoinvent values.